# Draft-04: Final XGBoost Fraud Pipeline (Kaggle)

This version is the final submit pipeline. It uses leakage-safe 5-fold CV, OOF threshold tuning for **Macro F1**, and fold-ensemble test predictions.


In [ ]:
import importlib.util
import os
import random
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

def run_cmd(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f'{pkg} is already installed')
        return
    run_cmd([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('xgboost')
ensure_package('dotenv', 'python-dotenv')
ensure_package('kaggle')

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


In [ ]:
COMPETITION = 'fraud-detection-cpe-232-data-models'
RUN_DOWNLOAD = True
RUN_SUBMISSION = True

ID_COL = 'id'
LABEL_COL = 'is_fraud'
TIME_COL = 'time_ind'
N_SPLITS = 5
RANDOM_STATE = 42
EPS = 1.0
THRESHOLD_GRID = np.linspace(0.001, 0.999, 999)

XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'n_estimators': 2000,
    'learning_rate': 0.04,
    'max_depth': 8,
    'min_child_weight': 5,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.0,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'early_stopping_rounds': 150,
}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

def resolve_data_paths_fallback():
    kaggle_candidates = [
        Path('/kaggle/input/fraud-detection-cpe-232-data-models'),
        Path('/kaggle/input/fraudulent-transaction-detect'),
        Path('/kaggle/input'),
    ]
    local_candidates = [
        Path.cwd(),
        Path.cwd() / 'kaggle/01-fraudulent-transaction-detect',
        Path('/Users/beam/Workspace/Course/my-cpe-lab/Y2/CPE232/kaggle/01-fraudulent-transaction-detect'),
    ]

    candidate_pairs = []
    for base in kaggle_candidates + local_candidates:
        candidate_pairs.extend([
            (base / 'train.csv', base / 'test.csv', base / 'sample_submission.csv'),
            (base / 'data' / 'train.csv', base / 'data' / 'test.csv', base / 'data' / 'sample_submission.csv'),
            (
                base / 'fraud-detection-cpe-232-data-models' / 'train.csv',
                base / 'fraud-detection-cpe-232-data-models' / 'test.csv',
                base / 'fraud-detection-cpe-232-data-models' / 'sample_submission.csv',
            ),
        ])
    for train_path, test_path, sample_path in candidate_pairs:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path
    raise FileNotFoundError('Could not find train/test/sample CSV files in Kaggle input or local folders')

def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f'{competition}.zip'
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print('Competition zip download complete')
        except Exception as download_error:
            print('Bulk download failed, fallback to per-file download:', download_error)
            for f in files_resp.files:
                file_path = data_dir / f.name
                if file_path.exists() and not force_download:
                    continue
                api_client.competition_download_file(
                    competition=competition,
                    file_name=f.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        if not extract_dir.exists() or not any(extract_dir.iterdir()):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract_dir)

    train_path = extract_dir / 'train.csv'
    test_path = extract_dir / 'test.csv'
    sample_path = extract_dir / 'sample_submission.csv'

    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        fallback_train = data_dir / 'train.csv'
        fallback_test = data_dir / 'test.csv'
        fallback_sample = data_dir / 'sample_submission.csv'
        if fallback_train.exists() and fallback_test.exists() and fallback_sample.exists():
            train_path, test_path, sample_path = fallback_train, fallback_test, fallback_sample
        else:
            raise FileNotFoundError('Could not resolve competition train/test/sample files')

    return train_path, test_path, sample_path, zip_path, extract_dir


In [ ]:
load_dotenv('.env', override=True)
os.environ.pop('KAGGLE_API_TOKEN', None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print('Authenticated user:', api.get_config_value('username'))
except Exception as auth_error:
    print('Kaggle API auth not ready:', auth_error)
    print('Falling back to existing /kaggle/input or local data if available')

if RUN_DOWNLOAD:
    if api is None:
        raise RuntimeError('RUN_DOWNLOAD=True but Kaggle API auth is not available')
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / 'data'
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = prepare_competition_data(
        api_client=api,
        competition=COMPETITION,
        data_dir=DATA_DIR,
        force_download=False,
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    BASE_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else BASE_DIR
OUTPUT_PATH = OUTPUT_DIR / 'submission_draft04_xgboost_kaggle_final.csv'
DIAGNOSTICS_PATH = OUTPUT_DIR / 'diagnostics_draft04_xgboost_kaggle_final.csv'
THRESHOLD_TABLE_PATH = OUTPUT_DIR / 'threshold_table_draft04_xgboost_kaggle_final.csv'
FOLD_TABLE_PATH = OUTPUT_DIR / 'fold_scores_draft04_xgboost_kaggle_final.csv'

config_df = pd.DataFrame([
    {
        'competition': COMPETITION,
        'seed': RANDOM_STATE,
        'n_splits': N_SPLITS,
        'threshold_grid_size': len(THRESHOLD_GRID),
        'run_download': RUN_DOWNLOAD,
        'run_submission': RUN_SUBMISSION,
        'output_file': str(OUTPUT_PATH),
        'xgb_learning_rate': XGB_PARAMS['learning_rate'],
        'xgb_max_depth': XGB_PARAMS['max_depth'],
        'xgb_min_child_weight': XGB_PARAMS['min_child_weight'],
    }
])
display(config_df)

print('Train path:', TRAIN_PATH)
print('Test path:', TEST_PATH)
print('Sample path:', SAMPLE_PATH)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

expected_train_cols = {
    'id',
    'time_ind',
    'transac_type',
    'amount',
    'src_acc',
    'src_bal',
    'src_new_bal',
    'dst_acc',
    'dst_bal',
    'dst_new_bal',
    'is_fraud',
    'is_flagged_fraud',
}
expected_test_cols = expected_train_cols - {LABEL_COL}

assert set(train_df.columns) == expected_train_cols, 'Train schema mismatch'
assert set(test_df.columns) == expected_test_cols, 'Test schema mismatch'
assert sample_submission.columns.tolist() == [ID_COL, LABEL_COL], 'Sample submission schema mismatch'

class_counts = train_df[LABEL_COL].value_counts().sort_index()
fraud_rate = float(train_df[LABEL_COL].mean())
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('Class counts (0, 1):')
print(class_counts.to_string())
print(f'Fraud rate: {fraud_rate:.8f} ({fraud_rate * 100:.4f}%)')
display(train_df.head(3))


In [ ]:
NUMERIC_SOURCE_COLS = ['amount', 'src_bal', 'src_new_bal', 'dst_bal', 'dst_new_bal']

def get_fill_values(fit_df: pd.DataFrame) -> dict[str, float]:
    medians = fit_df[NUMERIC_SOURCE_COLS].median(numeric_only=True).to_dict()
    return {k: float(v) for k, v in medians.items()}

def build_account_maps(fit_df: pd.DataFrame) -> dict[str, pd.Series]:
    src_group = fit_df.groupby('src_acc', observed=True).agg(
        src_txn_count=(ID_COL, 'size'),
        src_unique_dst=('dst_acc', 'nunique'),
    )
    dst_group = fit_df.groupby('dst_acc', observed=True).agg(
        dst_txn_count=(ID_COL, 'size'),
        dst_unique_src=('src_acc', 'nunique'),
    )
    return {
        'src_txn_count': src_group['src_txn_count'],
        'src_unique_dst': src_group['src_unique_dst'],
        'dst_txn_count': dst_group['dst_txn_count'],
        'dst_unique_src': dst_group['dst_unique_src'],
    }

def build_base_features(df: pd.DataFrame, fill_values: dict[str, float]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    amount = df['amount'].fillna(fill_values['amount']).astype('float64')
    src_bal = df['src_bal'].fillna(fill_values['src_bal']).astype('float64')
    src_new_bal = df['src_new_bal'].fillna(fill_values['src_new_bal']).astype('float64')
    dst_bal = df['dst_bal'].fillna(fill_values['dst_bal']).astype('float64')
    dst_new_bal = df['dst_new_bal'].fillna(fill_values['dst_new_bal']).astype('float64')
    time_ind = df[TIME_COL].astype('int64')

    out['transac_type'] = df['transac_type'].fillna('MISSING').astype(str)
    out['amount'] = amount
    out['src_bal'] = src_bal
    out['src_new_bal'] = src_new_bal
    out['dst_bal'] = dst_bal
    out['dst_new_bal'] = dst_new_bal
    out['is_flagged_fraud'] = df['is_flagged_fraud'].astype('float64')

    out['src_balance_diff'] = src_new_bal - src_bal
    out['dst_balance_diff'] = dst_new_bal - dst_bal
    out['expected_src_new'] = src_bal - amount
    out['src_error'] = out['expected_src_new'] - src_new_bal
    out['expected_dst_new'] = dst_bal + amount
    out['dst_error'] = out['expected_dst_new'] - dst_new_bal

    out['abs_src_error'] = np.abs(out['src_error'])
    out['abs_dst_error'] = np.abs(out['dst_error'])
    out['error_gap'] = out['abs_src_error'] - out['abs_dst_error']
    out['balance_shift_total'] = np.abs(out['src_balance_diff']) + np.abs(out['dst_balance_diff'])

    out['src_zero_before'] = (src_bal == 0).astype('float64')
    out['src_zero_after'] = (src_new_bal == 0).astype('float64')
    out['dst_zero_before'] = (dst_bal == 0).astype('float64')
    out['dst_zero_after'] = (dst_new_bal == 0).astype('float64')

    out['amount_to_src_balance'] = amount / (np.abs(src_bal) + EPS)
    out['amount_to_dst_balance'] = amount / (np.abs(dst_bal) + EPS)
    out['amount_to_src_new_balance'] = amount / (np.abs(src_new_bal) + EPS)
    out['amount_over_src_before'] = amount / (src_bal + EPS)
    out['amount_over_dst_before'] = amount / (dst_bal + EPS)
    out['amount_over_src_before_cap'] = np.clip(out['amount_over_src_before'], -10, 10)
    out['amount_over_dst_before_cap'] = np.clip(out['amount_over_dst_before'], -10, 10)

    out['emptied_account'] = ((src_bal > 0) & (src_new_bal == 0)).astype('float64')
    out['large_transfer'] = (amount > 200_000).astype('float64')
    out['is_round_amount'] = (amount % 1000 == 0).astype('float64')
    out['amount_bucket'] = pd.cut(
        amount,
        bins=[-np.inf, 100, 1_000, 10_000, 100_000, 200_000, 500_000, np.inf],
        labels=False,
    ).astype('float64')

    out['hour'] = (time_ind % 24).astype('float64')
    out['day'] = (time_ind // 24).astype('float64')
    out['is_night'] = out['hour'].isin([0, 1, 2, 3, 4, 5]).astype('float64')
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24.0)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24.0)
    out['log_amount'] = np.log1p(amount)

    src_prefix_merchant = df['src_acc'].astype(str).str.lower().str.startswith('mer')
    dst_prefix_merchant = df['dst_acc'].astype(str).str.lower().str.startswith('mer')
    prefix_merchant = src_prefix_merchant | dst_prefix_merchant
    missing_merchant_proxy = df[['dst_bal', 'dst_new_bal']].isna().any(axis=1)
    out['merchant_prefix_flag'] = prefix_merchant.astype('float64')
    out['merchant_missing_proxy'] = missing_merchant_proxy.astype('float64')
    out['merchant_indicator'] = np.where(prefix_merchant, 1.0, missing_merchant_proxy.astype(float))

    return out

def add_account_features(base_features: pd.DataFrame, df: pd.DataFrame, account_maps: dict[str, pd.Series]) -> pd.DataFrame:
    out = base_features.copy()
    out['src_txn_count'] = df['src_acc'].map(account_maps['src_txn_count']).fillna(0.0).astype('float64')
    out['src_unique_dst'] = df['src_acc'].map(account_maps['src_unique_dst']).fillna(0.0).astype('float64')
    out['dst_txn_count'] = df['dst_acc'].map(account_maps['dst_txn_count']).fillna(0.0).astype('float64')
    out['dst_unique_src'] = df['dst_acc'].map(account_maps['dst_unique_src']).fillna(0.0).astype('float64')
    return out

def make_model_matrix(fit_features: pd.DataFrame, transform_features: pd.DataFrame, encoder: ColumnTransformer | None = None):
    categorical_cols = ['transac_type']
    numeric_cols = [c for c in fit_features.columns if c not in categorical_cols]
    if encoder is None:
        encoder = ColumnTransformer(
            transformers=[
                ('transac_type_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
                ('numeric', 'passthrough', numeric_cols),
            ],
            remainder='drop',
            sparse_threshold=1.0,
        )
        fit_matrix = encoder.fit_transform(fit_features)
    else:
        fit_matrix = encoder.transform(fit_features)
    transform_matrix = encoder.transform(transform_features)
    return fit_matrix, transform_matrix, encoder

def has_inf_values(df: pd.DataFrame) -> bool:
    numeric_df = df.select_dtypes(include=[np.number])
    for col in numeric_df.columns:
        if np.isinf(numeric_df[col].to_numpy()).any():
            return True
    return False

def metric_at_threshold(y_true: np.ndarray, prob: np.ndarray, threshold: float) -> dict[str, float]:
    pred = (prob >= threshold).astype(int)
    macro_f1 = f1_score(y_true, pred, average='macro', zero_division=0)
    fraud_f1 = f1_score(y_true, pred, pos_label=1, zero_division=0)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'macro_f1': float(macro_f1),
        'fraud_f1': float(fraud_f1),
        'precision': float(precision),
        'recall': float(recall),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def threshold_search(y_true: np.ndarray, prob: np.ndarray, thresholds: np.ndarray):
    rows = []
    best_threshold = 0.5
    best_macro_f1 = -1.0
    for t in thresholds:
        m = metric_at_threshold(y_true, prob, float(t))
        rows.append({'threshold': float(t), **m})
        if m['macro_f1'] > best_macro_f1:
            best_macro_f1 = m['macro_f1']
            best_threshold = float(t)
    threshold_df = pd.DataFrame(rows).sort_values(['macro_f1', 'threshold'], ascending=[False, True])
    best_row = threshold_df.iloc[0].to_dict()
    return threshold_df, best_row

def get_feature_importance_df(model: XGBClassifier, encoder: ColumnTransformer) -> pd.DataFrame:
    booster = model.get_booster()
    score_map = booster.get_score(importance_type='gain')
    if not score_map:
        return pd.DataFrame({'feature': [], 'gain': []})
    names = list(encoder.get_feature_names_out())
    key_to_name = {f'f{i}': names[i] for i in range(len(names))}
    rows = []
    for k, v in score_map.items():
        rows.append({'feature': key_to_name.get(k, k), 'gain': float(v)})
    return pd.DataFrame(rows)


In [ ]:
X_test_fold_source = test_df.copy()
y = train_df[LABEL_COL].astype(int).to_numpy()
oof_prob = np.zeros(len(train_df), dtype='float64')
test_prob_folds = []
fold_rows = []
importance_rows = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df[LABEL_COL]), start=1):
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)
    y_tr = tr_df[LABEL_COL].astype(int).to_numpy()
    y_va = va_df[LABEL_COL].astype(int).to_numpy()

    fill_values = get_fill_values(tr_df)
    account_maps = build_account_maps(tr_df)

    X_tr_base = build_base_features(tr_df, fill_values)
    X_va_base = build_base_features(va_df, fill_values)
    X_te_base = build_base_features(X_test_fold_source, fill_values)

    X_tr = add_account_features(X_tr_base, tr_df, account_maps)
    X_va = add_account_features(X_va_base, va_df, account_maps)
    X_te = add_account_features(X_te_base, X_test_fold_source, account_maps)

    assert not has_inf_values(X_tr), f'Fold {fold}: inf values in train features'
    assert not has_inf_values(X_va), f'Fold {fold}: inf values in valid features'
    assert not has_inf_values(X_te), f'Fold {fold}: inf values in test features'

    X_tr_matrix, X_va_matrix, encoder = make_model_matrix(X_tr, X_va, encoder=None)
    _, X_te_matrix, _ = make_model_matrix(X_tr, X_te, encoder=encoder)

    va_probe = X_va.head(8).copy()
    va_probe.loc[:, 'transac_type'] = 'UNSEEN_TYPE'
    _ = encoder.transform(va_probe)

    pos_count = int(y_tr.sum())
    neg_count = int(len(y_tr) - pos_count)
    scale_pos_weight = float(neg_count / max(pos_count, 1))

    fold_model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=scale_pos_weight)
    fold_model.fit(
        X_tr_matrix,
        y_tr,
        eval_set=[(X_va_matrix, y_va)],
        verbose=False,
    )

    va_prob = fold_model.predict_proba(X_va_matrix)[:, 1]
    te_prob = fold_model.predict_proba(X_te_matrix)[:, 1]
    oof_prob[va_idx] = va_prob
    test_prob_folds.append(te_prob)

    fold_m = metric_at_threshold(y_va, va_prob, 0.5)
    best_iter = getattr(fold_model, 'best_iteration', None)
    if best_iter is None:
        best_iter = XGB_PARAMS['n_estimators'] - 1
    fold_rows.append({
        'fold': fold,
        'valid_rows': int(len(va_df)),
        'valid_fraud_rate': float(va_df[LABEL_COL].mean()),
        'macro_f1_at_0_5': fold_m['macro_f1'],
        'fraud_f1_at_0_5': fold_m['fraud_f1'],
        'precision_at_0_5': fold_m['precision'],
        'recall_at_0_5': fold_m['recall'],
        'best_iteration': int(best_iter + 1),
        'scale_pos_weight': scale_pos_weight,
    })

    fold_imp = get_feature_importance_df(fold_model, encoder)
    if not fold_imp.empty:
        fold_imp['fold'] = fold
        importance_rows.append(fold_imp)

    print(f'Fold {fold}/{N_SPLITS} done | Macro F1@0.5={fold_m["macro_f1"]:.6f}')

assert np.isfinite(oof_prob).all(), 'OOF probabilities contain non-finite values'
assert not np.isnan(oof_prob).any(), 'OOF probabilities contain NaN values'

fold_scores_df = pd.DataFrame(fold_rows)
display(fold_scores_df)

test_prob_mean = np.mean(np.column_stack(test_prob_folds), axis=1)


In [ ]:
threshold_df, best_row = threshold_search(y, oof_prob, THRESHOLD_GRID)
best_threshold = float(best_row['threshold'])

metrics_05 = metric_at_threshold(y, oof_prob, 0.5)
metrics_best = metric_at_threshold(y, oof_prob, best_threshold)

print('OOF metrics @ threshold=0.5')
print(metrics_05)
print('OOF metrics @ best threshold')
print({'threshold': best_threshold, **metrics_best})

assert metrics_best['macro_f1'] + 1e-12 >= metrics_05['macro_f1'], 'Best threshold should not underperform 0.5 on macro F1'

display(threshold_df.head(10))

if importance_rows:
    imp_all = pd.concat(importance_rows, ignore_index=True)
    imp_mean = (
        imp_all.groupby('feature', as_index=False)['gain']
        .mean()
        .sort_values('gain', ascending=False)
        .reset_index(drop=True)
    )
else:
    imp_mean = pd.DataFrame({'feature': [], 'gain': []})

print('Top feature importances (mean gain across folds):')
display(imp_mean.head(25))


In [ ]:
test_pred = (test_prob_mean >= best_threshold).astype(int)
submission = pd.DataFrame({ID_COL: test_df[ID_COL], LABEL_COL: test_pred})

assert len(submission) == len(test_df), 'Submission length mismatch'
assert submission.columns.tolist() == [ID_COL, LABEL_COL], 'Submission schema mismatch'
assert set(submission[LABEL_COL].unique()).issubset({0, 1}), 'Submission target must be binary'

submission.to_csv(OUTPUT_PATH, index=False)
threshold_df.to_csv(THRESHOLD_TABLE_PATH, index=False)
fold_scores_df.to_csv(FOLD_TABLE_PATH, index=False)

diagnostics = pd.DataFrame([
    {
        'train_rows': int(len(train_df)),
        'test_rows': int(len(test_df)),
        'train_fraud_rate': float(train_df[LABEL_COL].mean()),
        'oof_macro_f1_at_0_5': float(metrics_05['macro_f1']),
        'oof_macro_f1_best': float(metrics_best['macro_f1']),
        'oof_fraud_f1_best': float(metrics_best['fraud_f1']),
        'oof_precision_best': float(metrics_best['precision']),
        'oof_recall_best': float(metrics_best['recall']),
        'best_threshold': float(best_threshold),
        'predicted_fraud_rows_test': int(submission[LABEL_COL].sum()),
        'predicted_fraud_rate_test': float(submission[LABEL_COL].mean()),
        'fold_macro_f1_at_0_5_mean': float(fold_scores_df['macro_f1_at_0_5'].mean()),
        'fold_macro_f1_at_0_5_std': float(fold_scores_df['macro_f1_at_0_5'].std(ddof=0)),
    }
])
diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)
display(diagnostics)
display(submission.head())

print('Saved submission:', OUTPUT_PATH)
print('Saved diagnostics:', DIAGNOSTICS_PATH)
print('Saved threshold table:', THRESHOLD_TABLE_PATH)
print('Saved fold scores:', FOLD_TABLE_PATH)


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError('RUN_SUBMISSION=True but Kaggle API auth is not available')

    submit_message = (
        f'draft-04 xgboost final | macro_f1={metrics_best["macro_f1"]:.6f} '
        f'fraud_f1={metrics_best["fraud_f1"]:.6f} '
        f'threshold={best_threshold:.3f} '
        f'time={datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")}Z'
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print('Submission response:', response)
else:
    print('RUN_SUBMISSION is False. File is ready at', OUTPUT_PATH)


## Why draft-4 is stronger (student voice)

In draft-03, we used one validation split, so the score could move a lot depending on that split.
In draft-4, we switched to 5-fold stratified CV, so every row gets validated once and the OOF score is more stable.
We also tuned the threshold on OOF probabilities using Macro F1, which matches the competition metric.

The new interaction features help the model catch weird balance behavior, especially when amounts do not match expected before/after balances.
Account frequency features are still leakage-safe because each fold builds them from fold-train only, then maps to fold-valid/test.

After threshold tuning, precision and recall trade off more clearly than using 0.5 by default.
So this version is more stable for final submission and usually generalizes better to Kaggle test data.
